# Test K1 — does the certified feedback track a disclosed one? (registered; runs once)

Registered in `thesis/checks/prereg/K1_letf.json` before any of its data were downloaded. Single-stock leveraged and inverse ETFs must trade `A·L·(L−1)·R` dollars of their stock in the direction of its daily return; SEC filings disclose how much (`A`, `L`). K1 asks whether the feedback measured from **anonymous** trades near the close is larger in stocks with more disclosed rebalancing (Primary 1), and whether the certified margin falls with it (Primary 2).

**Runtime:** CPU only (no GPU). Cell 2 about 30–60 minutes (five SEC files, ~2 GB, and daily bars). Cell 4 is the long one: about 10–20 hours of downloads in total, **resumable** — if Colab disconnects, run cells 1, 3 and 4 again and it continues where it stopped. Cell 5 (the test) runs once, 10–30 minutes.

**Secrets** (Colab → key icon): `ALPACA_KEY`, `ALPACA_SECRET` (the free account used in Part 14), and `SEC_UA` = your name and e-mail, e.g. `Pratik Niroula research <you@example.com>` (the SEC requires a contact in the User-Agent). Never paste keys into a cell.

In [ ]:
# 1. Registered code, Drive folder, keys
import os, subprocess, sys, json, shutil, zipfile
from pathlib import Path
REPO = "/content/Fianince"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/Pratikn03/Fianince.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "pull", "-q", "origin", "main"], check=True)
sys.path.insert(0, f"{REPO}/thesis/evidence")
from google.colab import drive, userdata
drive.mount("/content/drive", timeout_ms=300000)
OUT = "/content/drive/MyDrive/k1_data"; os.makedirs(OUT, exist_ok=True)
import k1_letf as K
spec, sha, commit = K.gate()
print("specification sha256:", sha, "| registered in commit:", commit)
API = K.Alpaca(userdata.get("ALPACA_KEY"), userdata.get("ALPACA_SECRET"))
t = API.get(f"{API.DATA}/stocks/trades", {"symbols": "AAPL", "start": "2026-06-10T18:00:00Z", "end": "2026-06-10T18:00:05Z", "feed": "sip", "limit": 5})
assert "trades" in t and t["trades"].get("AAPL"), f"historical SIP trades not available on this account: {t}"
REPO_COUNTER = Path(REPO) / "thesis/checks/prereg/K1_unlock.json"; DRIVE_COUNTER = Path(OUT) / REPO_COUNTER.name
if not DRIVE_COUNTER.exists():
    shutil.copy(REPO_COUNTER, DRIVE_COUNTER)
print("counter:", json.loads(DRIVE_COUNTER.read_text()), "| data folder:", OUT)

In [ ]:
# 2. Universe, fixed before any trades are downloaded (refuses once cell 4 has started)
assert not (Path(OUT) / "K1_universe.lock").exists(), "the universe is frozen; skip this cell"
ZIPS = K.download_nport(OUT, userdata.get("SEC_UA"))
EXCLUSIONS = {}   # {fund series id: "reason"} ONLY for a name matched to the wrong company; decide on reading, before cell 3
U = K.universe(API, ZIPS, Path(OUT) / "K1_universe.json", exclusions=EXCLUSIONS)
import pandas as pd
tr = {s: u for s, u in U["units"].items() if u["role"] == "treated"}
print(f"pool {U['pool_size']} stocks | treated {len(tr)} | controls {len(U['units']) - len(tr)} | sample days {len(U['days'])}")
print(pd.DataFrame(tr).T[["Lambda", "ADV", "lambda"]].sort_values("lambda", ascending=False).to_string())
print("\nfund matches (check each name refers to the stock it is matched to):")
print(pd.DataFrame(U["funds"])[["SERIES_ID", "SERIES_NAME", "underlying", "L", "NET_ASSETS"]].to_string())

In [ ]:
# 3. Freeze the universe (after this the fund matches cannot change)
print("universe sha256:", K.lock_universe(OUT))

In [ ]:
# 4. Download the trades (resumable; rerun cells 1, 3, 4 after a disconnect)
info = K.collect(API, OUT)
print(info)

In [ ]:
# 5. RUN THE TEST (once; refuses until every stock-day is collected, and if it was opened before)
res = K.run(OUT, counter=DRIVE_COUNTER)

In [ ]:
# 6. Save the result: downloads k1_result.zip; send it in the chat
shutil.copy(DRIVE_COUNTER, REPO_COUNTER)
FILES = ["thesis/checks/prereg/K1_unlock.json", "thesis/evidence/K1_results.json", "thesis/checks/K1_numbers.tex"]
with zipfile.ZipFile("/content/k1_result.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in FILES:
        z.write(f"{REPO}/{f}", f)
    z.write(f"{OUT}/K1_universe.json", "thesis/evidence/K1_universe.json")
from google.colab import files
files.download("/content/k1_result.zip")